# Trabajo Práctico 2 (TP2) - Parte 2: Regresión Logística, Curva de Aprendizaje y Evaluación de Desempeño

### Machine Learning 1 (23433)
#### Facultad de Ingeniería - Universidad Nacional de Asunción (FIUNA)

---

## Objetivos de la Parte 2
1. Configurar una partición estratificada Train/Test (80% Train, 20% Test sobre 64,295 muestras) previniendo el fuga de datos (*data leakage*).
2. Diseñar un `ColumnTransformer` profesional con `SimpleImputer`, `PolynomialFeatures(degree=2)`, `StandardScaler` y `OneHotEncoder`.
3. Entrenar un modelo de `LogisticRegression(class_weight='balanced')` con optimización de hiperparámetros vía `GridSearchCV`.
4. Calcular y graficar la **Curva de Aprendizaje (`learning_curve`)** evaluando la convergencia en validación cruzada.
5. Optimizar el umbral de decisión ($	au^*$) e inspeccionar la Matriz de Confusión, Curva ROC (AUC) y Curva Precision-Recall.


In [27]:
# 1. Carga de Librerías y Dataset del TP2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Carga del Dataset del TP2
csv_path = 'reglamento_nuevo_unificado.csv'
df_raw = pd.read_csv(csv_path)
df_clean = pd.read_csv('df_clean.csv')
print(f"Dataset del TP2 cargado exitosamente: {df_raw.shape[0]:,} filas.")


Dataset del TP2 cargado exitosamente: 64,295 filas.


## Ejercicio 1: Partición Estratificada Train / Test

Define las listas de atributos numéricos `num_features` y categóricos `cat_features`.
Realiza una partición estratificada del 80% para entrenamiento (`X_train`) y 20% para prueba (`X_test`).


In [28]:


# TODO: Definir num_features, cat_features y particionar X e y
# === ESCRIBE TU CÓDIGO AQUÍ ===

num_features = ['Primer_Par_Clean', 'Segundo_Par_Clean', 'Score_Parciales', 'Diff_Parciales', 'TPLab_Clean', 'Firma_Clean', 'FirmaCalc_Clean', 'Asis_Clean']
cat_features = ['Carrera_Nombre', 'Semestre']

# Suponiendo df_clean cargado del notebook anterior
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X = df_clean[num_features + cat_features]
y = df_clean['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test:  {X_test.shape}")
print(f"Distribución de clases en Train (proporción): {y_train.value_counts(normalize=True).to_dict()}")
print(f"Distribución de clases en Test (proporción):  {y_test.value_counts(normalize=True).to_dict()}")


Dimensiones de X_train: (51436, 10)
Dimensiones de X_test:  (12859, 10)
Distribución de clases en Train (proporción): {1: 0.6066568162376546, 0: 0.39334318376234545}
Distribución de clases en Test (proporción):  {1: 0.6066568162376546, 0: 0.39334318376234545}


## Ejercicio 2: Construcción del Preprocesador Profesional (`ColumnTransformer`)

Construye un `Pipeline` numérico con:
1. `SimpleImputer(strategy='median')`
2. `PolynomialFeatures(degree=2, include_bias=False)`
3. `StandardScaler()`

Y un `Pipeline` categórico con `SimpleImputer` y `OneHotEncoder(handle_unknown='ignore')`.


In [ ]:
# TODO: Ensamblar num_transformer, cat_transformer y preprocessor
# === ESCRIBE TU CÓDIGO AQUÍ ===
num_transformer = Pipeline([  
    ('imputer', SimpleImputer(strategy='median')),  #Rellena valores nulos
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), #Imputa datos faltantes
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
]) #Convierte columnas categóricas en columnas binarias

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])  #Unifica los pipelines numéricos y categóricos en un solo objeto

X_train_prep = preprocessor.fit_transform(X_train) #Verifica que genera correctamente las 53 columnas
feature_names = preprocessor.get_feature_names_out()
print(f" Preprocesador ensamblado con éxito.")
print(f"Dimensiones de X_train original:    {X_train.shape}")
print(f"Dimensiones de X_train transformado: {X_train_prep.shape}")
print(f"Total de características generadas:  {len(feature_names)}")

 Preprocesador ensamblado con éxito.
Dimensiones de X_train original:    (51436, 10)
Dimensiones de X_train transformado: (51436, 53)
Total de características generadas:  53


## Ejercicio 3: Entrenamiento del Modelo y Búsqueda en Malla (`GridSearchCV`)

Ensambla `full_pipeline` con `LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=2000)`.
Configura `GridSearchCV` sobre el parámetro de regularización `C` en `[0.01, 0.1, 1.0, 10.0]` con `scoring='f1'`.


In [ ]:
# TODO: Instanciar full_pipeline y GridSearchCV
# === ESCRIBE TU CÓDIGO AQUÍ ===
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='lbfgs', max_iter=2000, random_state=42, class_weight='balanced'))
])

param_grid = {'classifier__C': [0.01, 0.1, 1.0, 10.0]}
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1,  # Utiliza todos los núcleos de procesador disponibles
    verbose=1
)
print("Iniciando la búsqueda en malla (GridSearchCV)...")
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print("\n Búsqueda completada exitosamente.")
print(f"Mejor valor de C encontrado: {grid_search.best_params_['classifier__C']}")
print(f"Mejor F1-Score promedio en Validación Cruzada (CV): {grid_search.best_score_:.4f}")
# 6. Tabla detallada de resultados para cada hiperparámetro C
cv_results = pd.DataFrame(grid_search.cv_results_)[
    ['param_classifier__C', 'mean_test_score', 'std_test_score', 'rank_test_score']
]
cv_results.columns = ['Parámetro C', 'F1-Score Medio (CV)', 'Desv. Estándar', 'Ranking']
display(cv_results.sort_values(by='Ranking'))


## Ejercicio 4: Graficación de la Curva de Aprendizaje (`learning_curve`)

Calcula y grafica la curva de aprendizaje utilizando `learning_curve` sobre `X_train` y `y_train` con 10 tamaños de entrenamiento (`np.linspace(0.1, 1.0, 10)`).


In [ ]:
# 1. Cálculo de la Curva de Aprendizaje sobre X_train y y_train
train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_model,
    X=X_train,
    y=y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)
# 2. Promedios y Desviaciones Estándar por tamaño de muestra
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)
# 3. Graficación de las Curvas de Entrenamiento y Validación
plt.figure(figsize=(10, 6))
# Curva de Entrenamiento
plt.plot(train_sizes, train_mean, 'o-', color='#1f77b4', linewidth=2, label='F1-Score (Entrenamiento)')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#1f77b4')
# Curva de Validación Cruzada
plt.plot(train_sizes, val_mean, 'o-', color='#d62728', linewidth=2, label='F1-Score (Validación Cruzada)')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#d62728')
# Estética y Etiquetas
plt.title('Curva de Aprendizaje (Learning Curve) - Regresión Logística', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de Muestras de Entrenamiento', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.ylim([0.7, 1.0])
plt.legend(loc='lower right', fontsize=11, frameon=True)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Ejercicio 5: Optimización del Umbral de Decisión y Evaluación de Desempeño

En el conjunto de prueba (`X_test`):
1. Obtén las probabilidades de aprobación `y_proba_test`.
2. Optimiza el umbral de decisión ($	au^*$) entre `0.1` y `0.9` que maximiza el F1-Score.
3. Grafica la **Matriz de Confusión**, la **Curva ROC (AUC)** y la **Curva Precision-Recall**.


In [ ]:
# 1. Obtener probabilidades en el conjunto de test
y_proba_test = best_model.predict_proba(X_test)[:, 1]
# 2. Búsqueda del umbral tau* que maximiza el F1-Score entre 0.1 y 0.9
thresholds = np.linspace(0.1, 0.9, 81)
f1_scores = [f1_score(y_test, (y_proba_test >= t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]
best_f1_test = f1_scores[best_idx]
print(f" Umbral Óptimo (tau*): {best_thresh:.2f}")
print(f" F1-Score en Test con tau*: {best_f1_test:.4f}\n")
# Predicciones con el umbral optimizado
y_pred_opt = (y_proba_test >= best_thresh).astype(int)
# 3. Graficación: Matriz de Confusión, Curva ROC y Precision-Recall
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# A. Matriz de Confusión
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title(f'Matriz de Confusión (tau = {best_thresh:.2f})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')
# B. Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_test)
auc_score = roc_auc_score(y_test, y_proba_test)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {auc_score:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
axes[1].set_title('Curva ROC', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.6)
# C. Curva Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_proba_test)
ap_score = average_precision_score(y_test, y_proba_test)
axes[2].plot(recall, precision, color='green', lw=2, label=f'PR (AP = {ap_score:.4f})')
axes[2].set_title('Curva Precision-Recall', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(loc='lower left')
axes[2].grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
# Reporte de Clasificación detallado
print("=== REPORTE DE CLASIFICACIÓN CON UMBRAL ÓPTIMO ===")
print(classification_report(y_test, y_pred_opt, digits=4))